In [1]:
import lfox
import lfox.lattice as lat
import lfox.evolution.hmc as lhmc
import jax
import jax.numpy as jnp
import numpy as np

import lfox.fermions.spin as spin

# Imports below require "dev" environment
import matplotlib.pyplot as plt
import lsqfit
import gvar as gv
import tqdm

# Double precision!
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_threefry_partitionable", True)

In [2]:
Lat4 = lat.SquareLattice(dims=(4,4,8))
t1 = lat.LatticeTensorField(Lat4, [('spin', 4)])
print(t1.dims)
t1.F = t1.F.at[0,0,0].set(1)
print(t1.F[0,0,0])

(4, 4, 8, 4)
[1. 1. 1. 1.]


I0000 00:00:1696476918.469981       1 tfrt_cpu_pjrt_client.cc:349] TfrtCpuClient created.


In [3]:
t1.labels

{'spin': 3}

In [4]:
si = t1.labels['spin']
jnp.take(t1.F, 0, axis=si)

Array([[[1., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.]],

       [[0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.]],

       [[0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.]],

       [[0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.]]], dtype=float64)

In [5]:
Pauli_X = np.array([[0,1],[1,0]])
Pauli_Y = np.array([[0,-1j],[1j,0]])
Pauli_Z = np.array([[1,0],[0,-1]])

In [6]:
# DeGrand-Rossi gamma basis

Gamma_0 = np.array([
    [0, 0, 0, 1j],
    [0, 0, 1j, 0],
    [0, -1j, 0, 0],
    [-1j, 0, 0, 0],
])
Gamma_1 = np.array([
    [0, 0, 0, -1],
    [0, 0, 1, 0],
    [0, 1, 0, 0],
    [-1, 0, 0, 0],
])
Gamma_2 = np.array([
    [0, 0, 1j, 0],
    [0, 0, 0, -1j],
    [-1j, 0, 0, 0],
    [0, 1j, 0, 0],
])
Gamma_3 = np.array([
    [0, 0, 1, 0],
    [0, 0, 0, 1],
    [1, 0, 0, 0],
    [0, 1, 0, 0],
])
Gamma_5 = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, -1, 0],
    [0, 0, 0, -1],
])

In [7]:
# Euclidean product of all four gammas --> gamma_5
np.all(Gamma_0 @ Gamma_1 @ Gamma_2 @ Gamma_3 == Gamma_5)

True

In [8]:
Lat4 = lat.SquareLattice(dims=(2,3))
psi = spin.Dirac4DFermionField(Lat4).unit_fill()
chi = spin.Dirac4DFermionField(Lat4).unit_fill()


In [9]:
(psi.conj() * chi).F

Array([[[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]],

       [[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]]], dtype=float64)

In [10]:
M = np.array([[1,2],[3,4]])
np.einsum('...i',M)

array([[1, 2],
       [3, 4]])

In [11]:
print(psi.F.shape)
print(psi.dims, psi.labels)

(2, 3, 4)
(2, 3, 4) {'spin': 2}


In [12]:
180/4

45.0

In [13]:
def spin_gamma_product(psi_L, Gamma, psi_R):
    return jnp.einsum('i,ij,j', psi_L.conj(), Gamma, psi_R)

@jax.jit
def gp2(psi_L, Gamma, psi_R):
    return jnp.einsum('...i,ij,...j', psi_L.conj(), Gamma, psi_R)

In [14]:
print(spin_gamma_product(np.ones(4), spin.GammaMatrix[5], np.ones(4)))

0.0


In [15]:
vv = lambda x, y: jnp.vdot(x,y)
mv = jax.vmap(vv, (0, None), 0)
mm = jax.vmap(mv, (None, 1), 1)
x = jnp.ones((2,3))
y = jnp.ones((3,4))
z = jnp.ones(3)

print(vv(z,z))
print(mv(x,z))

3.0
[3. 3.]


In [16]:
M = jax.vmap(vv, 0, 0)
M(z,z)

Array([1., 1., 1.], dtype=float64)

In [17]:
lat_sgp = jax.vmap(jax.vmap(spin_gamma_product, (0, None, 0)), (0, None, 0))

In [18]:
%time lat_sgp(psi.F, spin.GammaMatrix[5], chi.F)

CPU times: user 38 ms, sys: 2.44 ms, total: 40.4 ms
Wall time: 38.9 ms


Array([[0., 0., 0.],
       [0., 0., 0.]], dtype=float64)

In [19]:
lat_sgp_jit = jax.jit(lat_sgp)

In [54]:
%time lat_sgp_jit(psi.F, spin.GammaMatrix[5], chi.F)

CPU times: user 328 µs, sys: 185 µs, total: 513 µs
Wall time: 344 µs


Array([[0., 0., 0.],
       [0., 0., 0.]], dtype=float64)

In [55]:
%time gp2(psi.F, spin.GammaMatrix[5], chi.F)

CPU times: user 252 µs, sys: 181 µs, total: 433 µs
Wall time: 248 µs


Array([[0., 0., 0.],
       [0., 0., 0.]], dtype=float64)

In [24]:
print(psi.bilinear(chi).F)
print(psi.bilinear(chi, spin_mat=spin.GammaMatrix[5]).F)

[[4. 4. 4.]
 [4. 4. 4.]]
[[0. 0. 0.]
 [0. 0. 0.]]


In [57]:
%time psi.bilinear(chi, spin_mat=spin.GammaMatrix[5])

CPU times: user 289 µs, sys: 93 µs, total: 382 µs
Wall time: 346 µs


In [31]:
psi._tree_flatten()

((Array([[[1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.]],
  
         [[1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.]]], dtype=float64),),
 {'lattice': <lfox.lattice.SquareLattice at 0x127bdaad0>,
  'bc': array([1., 1.])})